# 03 | Histórico sintético por coorte e disciplina

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** gerar histórico reproduzível condicionado à estrutura importada. **Entradas:** seed42, cenário baseline, ciclos2022-1 a2024-1; alvo2024-2. **Esperado:** snapshot SQL imutável, planejado separado de sintético.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from pathlib import Path
from src.academic.importer import import_sources
from src.simulation.academic import simulate
from src.simulation.store import save
SEED=42
CENARIO='baseline'
snapshot=simulate(import_sources(Path('/reference')),seed=SEED,profile=CENARIO)
save(snapshot)
save_context(snapshot_id=snapshot['snapshot_id'])
display({k:snapshot[k] for k in ('snapshot_id','dataset_sha256','scenario_id','history_window','target','data_completeness')})


**Executar:** observar regulares e adicionais por ciclo. **Resultado:** linha temporal SYNTHETIC_HISTORY; alvo futuro não entra no treino.

In [ ]:

from src.workbench.academic_data import load
frame,_=load(snapshot['snapshot_id'])
display(frame.groupby(['course_code','split']).size().rename('observações').reset_index())
cyber=frame[(frame.course_code=='2287') & (frame.period==2)]
display(cyber[['cycle_code','cohort_id','discipline_code','regular','dependency','elective','split']].tail(12))
cyber.groupby('cycle_code')[['regular','dependency','elective']].mean().plot(marker='o',title='Médias DEMO por disciplina — não matrícula real')


**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.